In [ ]:
"""
Deepseek v4.1 introduced Compressed Sparse Attention 2 (CSA2) which intends to reduce 
the kv cache storage and attention computation simultaneously.
(https://arxiv.org/pdf/2609.19969)
"""

In [14]:
import math
import torch
from torch import nn
import torch.nn.functional as f


class Main_KV:

    def __init__(self, d_latent=16):
        self.d_latent = d_latent
        self.main_kv = None

    def append(self, new_latent):
        if self.main_kv == None:
            self.main_kv = new_latent
        else:
            # B, S, D
            self.main_kv = torch.cat(
                [self.main_kv, new_latent], dim=1
            )  # B, S, d_latent

    def get_topk(self, topk_indices):
        B, H, K = topk_indices.shape
        expanded = self.main_kv.unsqueeze(1).expand(-1, H, -1, -1)
        idx_expanded = topk_indices.unsqueeze(-1).expand(
            -1, -1, -1, self.d_latent
        )
        return torch.gather(expanded, dim=2, index=idx_expanded)


class SWA_KV:

    def __init__(self, window_size=32, batch=32, kv_heads=2, head_dim=128):
        self.window_size = window_size

        # B, heads, 0, head_dim
        self.k = torch.empty((batch, kv_heads, 0, head_dim))
        self.v = torch.empty((batch, kv_heads, 0, head_dim))

    def update(self, new_k, new_v):
        # k_new: B, heads, 1, head_dim
        self.k = torch.cat([self.k, new_k], dim=2)
        self.v = torch.cat([self.v, new_v], dim=2)

        # only keep the windowlength
        self.k = self.k[:, :, -self.window_size :, :]
        self.v = self.v[:, :, -self.window_size :, :]

        return self.k, self.v


class CSA_2(nn.Module):

    def __init__(
        self,
        D=1024,
        mode="full",
        heads=2,
        d_latent=16,
        d_ind=8,
        top_k=4,
        batch=1,
    ):
        """
        mode: full, reindex, reuse
        """
        super().__init__()
        assert mode in ["full", "reindex", "reuse"]
        assert D % heads == 0

        self.mode = mode
        self.head_dim = D // heads
        self.d_ind = d_ind
        self.n_heads = heads
        self.D = D
        self.top_k = top_k
        self.d_latent = d_latent

        self.wq = nn.Linear(D, D)
        self.wk = nn.Linear(D, D)
        self.wv = nn.Linear(D, D)

        self.swa_kv_cache = SWA_KV(
            kv_heads=heads, head_dim=self.head_dim, batch=batch
        )

        if self.mode == "full":
            self.compress_proj = nn.Linear(D, d_latent)
        if self.mode in ["full", "reindex", "reuse"]:
            self.k_indexer = nn.Linear(d_latent, d_ind)
            self.q_indexer = nn.Linear(self.head_dim, d_ind)

        self.w_uk = nn.Linear(d_latent, self.head_dim)
        self.w_uv = nn.Linear(d_latent, self.head_dim)

        self.wo = nn.Linear(D, D)

    def split_heads(self, x):
        # x: B, S, D
        x = x.view(
            x.shape[0], x.shape[1], self.n_heads, self.head_dim
        )  # B, S, H, head_dim
        return x.transpose(1, 2)  # B, H, S, head_dim

    def forward(self, x, global_main_kv=None, cached_topk_indices=None):
        # x: B, S, D
        q = self.wq(x)
        x_last = x[:, -1:, :]  # B, 1, D

        if global_main_kv == None:
            global_main_kv = Main_KV(d_latent=self.d_latent)

        # q: B, H, S, head_dim
        q = self.split_heads(self.wq(x_last))  # B, H, 1, head_dim
        new_k = self.split_heads(self.wk(x_last))  # B, 1, D -> B, H, 1, h_dim
        new_v = self.split_heads(self.wv(x_last))

        k_swa, v_swa = self.swa_kv_cache.update(new_k, new_v)

        if self.mode == "full":
            # k, q indexer first
            # selector
            c_kv = self.compress_proj(x_last)
            global_main_kv.append(c_kv)

        if self.mode in ["full", "reindex"]:
            q_ind = self.q_indexer(q)  # B, H, 1, d_ind
            k_ind = self.k_indexer(
                global_main_kv.main_kv
            )  # B, S_hist, d_latent

            k_ind = k_ind.transpose(-1, -2).unsqueeze(1)  # B, 1, d_ind, S_hist

            # B, H, 1, d_ind @ B, 1, d_ind, S_hist -> B, H, 1, S_hist
            scores = q_ind @ k_ind
            scores = scores.squeeze(2)  # B, H, S_hist

            k_val = min(self.top_k, scores.shape[-1])
            topk_indices = torch.topk(scores, k=k_val, dim=-1).indices  # B, H, k_val

        else:  # "reuse" mode
            topk_indices = cached_topk_indices

        selected_latents = global_main_kv.get_topk(topk_indices)
        k_main = self.w_uk(selected_latents)
        v_main = self.w_uv(selected_latents)  # B, H, K, head_dim

        full_k = torch.cat([k_swa, k_main], dim=2)
        full_v = torch.cat([v_swa, v_main], dim=2)

        # q: B, H, 1, hdim
        # kv: B, H, k+swa, hdim
        qk = (q @ full_k.transpose(-1, -2)) / torch.sqrt(
            torch.tensor(q.shape[-1], dtype=torch.float32)
        )  # B, H, 1, k+swa

        scores = f.softmax(qk, dim=-1)  # B, H, 1, k+swa

        attn = scores @ full_v  # B, H, 1, h_dim

        attn = attn.transpose(1, 2).contiguous()
        attn = attn.view(attn.shape[0], attn.shape[1], self.D)

        return self.wo(attn), global_main_kv, topk_indices


layer = CSA_2(D=64, batch=1)
x_test = torch.randn(1, 1, 64)
x = layer(x_test)
print("Output shape:", x[0].shape)

Output shape: torch.Size([1, 1, 64])


In [ ]:
"""
What is an indexer?
"""
# we use a toy example without batch dim
# s_hist is smaller than s because s counts every token while s_hist combines for example every 4 tokens into a latent. 
# q comes directly from x -wq-> q -split_heads-> q 
# kv comes from main_kv which has a specific latent_dim
q = torch.randn((32, 10))  # s, head_dim (queries)
kv = torch.randn((8, 10))  # s_hist, latent_dim (historical latents)

# we need query + latent -> selection scores over s
# actually projecting both to a shared low-dim indedx space
q_indexer = torch.nn.Linear(10, 4)  # d -> d_ind
k_indexer = torch.nn.Linear(10, 4)  # d -> d_ind

# here we project our x and main kv into indexer space 
q_ind = q_indexer(q)  # s, d -> s, d_ind
k_ind = k_indexer(kv)  # s_hist, d -> s_hist, d_ind

# we do a dotproduct to find similairities between vecotrs and extract the topk highest scores with torch.topk
scores = q_ind @ k_ind.T  # (t, d_ind) @ (d_ind, s) -> t, s
idx = torch.topk(scores, k=4, dim=-1).indices  # t, top_k

scores.shape  # t, s
idx.shape  # hell yeah

torch.Size([30])